# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

In [ ]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

In [ ]:
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.35.1


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [ ]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)
MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    52 seconds ago            


### **CLIENTE DE PYTHON**

In [ ]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)

print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
**Diferencia entre RAM y Almacenamiento**

En una computadora, la RAM (Random Access Memory) y el almacenamiento son dos componentes fundamentales que trabajan juntos para permitir que la computadora procese información.

**RAM (Random Access Memory)**

La RAM es un tipo de memoria que almacena datos y programas en la memoria principal de la computadora. Su función es proporcionar acceso rápido y directo a la información necesaria para que la computadora pueda procesarla. La RAM es como un "trayectorio" de información, donde la computadora puede acceder y modificar datos rápidamente.

**Almacenamiento**

El almacenamiento, por otro lado, es un componente que almacena datos y programas en dispositivos como discos duros, unidades de estado sólido (SSD), tarjetas de memoria externas, etc. El almacenamiento proporciona espacio para almacenar grandes cantidades de datos y programas, pero puede ser más lento que la R

In [ ]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "model": "llama3.2:3b",
  "created_at": "2026-10-06T02:30:45.006877411Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 103114418041,
  "load_duration": 56996235993,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 39775487000,
  "eval_count": 509,
  "eval_duration": 6338502000,
  "message": {
    "role": "assistant",
    "content": "**Diferencia entre RAM y Almacenamiento**\n\nEn una computadora, la RAM (Random Access Memory) y el almacenamiento son dos componentes fundamentales que trabajan juntos para permitir que la computadora procese información.\n\n**RAM (Random Access Memory)**\n\nLa RAM es un tipo de memoria que almacena datos y programas en la memoria principal de la computadora. Su función es proporcionar acceso rápido y directo a la información necesaria para que la computadora pueda procesarla. La RAM es como un \"trayectorio\" de información, donde la computadora puede acceder y modificar datos rápidamente.\n\n**Almacenamiento**\n\nEl almacenamiento

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)

Tokens del prompt: 43
Tokens de la respuesta: 509
Tokens totales: 552


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
inicio = time.time()
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 12.19 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)

Modelo ligero (llama3.2:3b): 12.19 s — 552 tokens
Modelo grande (llama3.1:8b): 27.92 s — 298 tokens (de los cuales 16.67 s fueron carga del modelo)

Respuesta del modelo grande:
 La diferencia entre la RAM (Memoria de Acceso Aleatorio) y el almacenamiento en una computadora se refiere a la función y el propósito de cada componente. 

La RAM es un tipo de memoria temporal donde la computadora almacena datos y programas que están siendo utilizados en ese momento. Los datos se cargan en la RAM cuando la computadora se enciende y se ejecutan mientras la computadora está en funcionamiento. 

El almacenamiento, por otro lado, es donde la computadora almacena datos permanentemente. Puede ser en forma de disco duro (HDD), disco sólido (SSD), unidad de estado sólido (SSD), unidad de estado sólido (SSD), disco Blu-ray (BD), etc. 

Los datos en el almacenamiento permanecen ahí incluso cuando la computadora se apaga. 

Por ejemplo, si alguien quiere abrir un programa en su computadora, primero lo 

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets
from google.colab import userdata
from google.colab import userdata
!pip install groq --quiet
from groq import Groq

api_key = userdata.get("GROQ_API_KEY")

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)


In [ ]:
models = client.models.list()

for model in models.data:
    print(model.id)

whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
allam-2-7b
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
whisper-large-v3


In [ ]:
# Definir la lista de preguntas
preguntas =[
    "¿Qué es la colorimetría?",
    "¿Cuál es la relación entre la música y las emociones?",
    "¿En qué año comenzó el estudio de redes neuronales?"

]

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

'¿Qué es la colorimetría?'

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
resultado_1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": preguntas[0]
        }
    ]
)

print(resultado_1.choices[0].message.content)

La colorimetría es la rama de la ciencia que estudia la medición y descripción del color de una manera cuantitativa. Se trata de un conjunto de métodos y estándares que permiten representar, comparar y reproducir colores de forma objetiva y reproducible, tanto en la práctica artística y de diseño como en la industria, la investigación y la tecnología.  

### 1. Conceptos básicos

| Concepto | Descripción |
|----------|-------------|
| **Color** | Percepción visual que resulta de la interacción de la luz con los objetos y la retina. |
| **Espacio de color** | Sistema de coordenadas que asigna valores numéricos a los colores. Ejemplos: RGB (rojo, verde, azul), CMYK (cian, magenta, amarillo, negro), CIE XYZ, LAB. |
| **CIE** | Comisión Internacional de Iluminación (International Commission on Illumination), que define los estándares y modelos de color más usados. |
| **Pantone** | Sistema comercial de correspondencia de colores usado en impresión y diseño. |

### 2. Principios de medición

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
resultado_2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": preguntas[1]
        }
    ]
)

print(resultado_2.choices[0].message.content)

## La música y las emociones: una relación profunda y bidireccional

La música es una de las formas de expresión humana más antiguas y universalmente reconocidas. Su capacidad para generar, intensificar, moderar e incluso transformar nuestras emociones la convierte en un fenómeno fascinante que ha sido objeto de estudio en psicología, neurociencia, antropología y arte. A continuación se detalla cómo la música y las emociones interactúan, los mecanismos involucrados y ejemplos concretos de su aplicación práctica.

---

### 1. Mecanismos neurobiológicos

| Proceso | Qué ocurre en el cerebro | Emoción asociada |
|---------|-------------------------|-----------------|
| **Detección de patrones** | Corteza auditiva procesa ritmo, tono, timbre | Curiosidad, sorpresa |
| **Asociación y memoria** | Hipocampo y corteza prefrontal integran experiencias pasadas | Nostalgia, alegría |
| **Liberación de neurotransmisores** | Dopamina, oxitocina, serotonina se liberan durante la escucha | Placer, em

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
resultado_3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": preguntas[2]
        }
    ]
)

print(resultado_3.choices[0].message.content)

El estudio formal de las redes neuronales comenzó en 1943, cuando los neurólogos y matemáticos Warren McCulloch y Walter Pitts publicaron su influyente artículo **“A Logical Calculus of the Ideas Immanent in Nervous Activity”**. En ese trabajo propusieron el primer modelo matemático de un “neurona artificial”, sentando las bases teóricas de lo que hoy conocemos como redes neuronales.  

Posteriormente, en la década de 1950, se desarrollaron algoritmos prácticos como el perceptrón de Frank Rosenblatt (1957), pero la idea de modelar la actividad cerebral con unidades conectadas empezó con McCulloch y Pitts en 1943.


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados =[resultado_1, resultado_2, resultado_3]

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
resultados

[ChatCompletion(id='chatcmpl-6bc37591-441e-422b-80cf-a1a5a21a2b25', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='La colorimetría es la rama de la ciencia que estudia la medición y descripción del color de una manera cuantitativa. Se trata de un conjunto de métodos y estándares que permiten representar, comparar y reproducir colores de forma objetiva y reproducible, tanto en la práctica artística y de diseño como en la industria, la investigación y la tecnología.  \n\n### 1. Conceptos básicos\n\n| Concepto | Descripción |\n|----------|-------------|\n| **Color** | Percepción visual que resulta de la interacción de la luz con los objetos y la retina. |\n| **Espacio de color** | Sistema de coordenadas que asigna valores numéricos a los colores. Ejemplos: RGB (rojo, verde, azul), CMYK (cian, magenta, amarillo, negro), CIE XYZ, LAB. |\n| **CIE** | Comisión Internacional de Iluminación (International Commission on Illumination), que def

# **De acuerdo a lo observado el modelo pequeño pudo contestar todas las preguntas realizadas**